# Preprocessing pipeline (P2P + SN2N)

This notebook does three things:
1) **Create fixed train/val/test split CSVs**
2) **Create preprocessed folders** using the CSV and augmentation pipeline
3) **Create a GT-prediction mapping**

**Fixed split rule**
- **TRAIN**: all patterns from simulation **07**
- **VAL1**:   half of patterns from simulation **08**
- **VAL1**: Same as VAL2, but processed similar to the test set so we can compare metrics 
- **TEST**:  other half of patterns from simulation **08**

> Tip: set `BASE_DIR` once in the parameters cell below.



In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Tuple


# --- Paths ---
BASE_DIR = Path(r"C:\Users\ntpar\Downloads\SN2N_Capstone")  # Change this to the base directory if you're using a folder for the project
DATA_DIR = BASE_DIR / "data"                           # simulations 07_*, 08_* live here
SPLITS_DIR = BASE_DIR / "splits"                        # where train/val/test CSVs go
OUT_DIR = SPLITS_DIR / "preprocessed-p2p"              # output folders {train, val1, val2, test}


# --- Split creation ---
SEED = 123
OVERWRITE_SPLITS = True


# --- Preprocessing / augmentation ---
OVERWRITE_PREPROCESSED = True
VERBOSE = True

# Percentile normalization used by make_sn2n_sample (train/val) and in test normalization
P_LOW = 1.0
P_HIGH = 99.5

# Basic augmentation (flips/rotations) probability
P_BASIC_AUG = 0.8

# Patch2Patch config
P2P_MODE = 2
P2P_PATCH_SIZE: Tuple[int, int] = (64, 64)
P2P_UP = 5  # saves 1+P2P_UP samples per input file (replicates datagen behaviour)

# Sliding-window patch extraction
USE_SLIDING = True
SLIDE_PATCH_SIZE: Tuple[int, int] = (128, 128)
SLIDE_STRIDE = 64


## 1) Create split CSVs

Creates `train.csv`, `val.csv`, `test.csv` in `SPLITS_DIR`.

Each CSV row corresponds to one **pattern** folder and includes the list of SOFI c2 files (semicolon-separated).


In [ ]:
from __future__ import annotations
import pandas as pd
import numpy as np
import tifffile as tiff
import csv
import random
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional


@dataclass(frozen=True)
class PatternItem:
    sim_id: str
    pattern_id: str
    pattern_dir: Path
    gt_path: Path
    noisy_dir: Path
    sofi_run_dir: Path
    sofi_c2_files: List[Path]


def _is_pattern_dir(p: Path) -> bool:
    return p.is_dir() and p.name.endswith("_Training_")


def _find_sim_dir(base: Path, sim_id: str) -> Path:
    matches = [p for p in base.iterdir() if p.is_dir() and p.name.split("_", 1)[0] == sim_id]
    if not matches:
        raise FileNotFoundError(f"No simulation folder found for sim_id='{sim_id}' in {base}")
    if len(matches) > 1:
        matches.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    return matches[0]


def _pick_sofi_run_dir(sofi_root: Path, prefer_contains: str = "1_100f") -> Optional[Path]:
    if not sofi_root.exists():
        return None
    candidates = [d for d in sofi_root.iterdir() if d.is_dir()]
    if not candidates:
        return None
    preferred = [d for d in candidates if prefer_contains in d.name]
    if preferred:
        preferred.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        return preferred[0]
    candidates.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    return candidates[0]


def _find_gt_file(pattern_dir: Path) -> Path:
    """Robust GT lookup (handles small filename variations)."""
    gt_dir = pattern_dir / "GT" / "ConvDownUp" / "Rescaled_GT"
    # common exact names seen in your notebook
    candidates = [
        gt_dir / "2o_Training_GTconvres_135nm.tif",
        gt_dir / "2o_Training__GTconvres_135nm.tif",
    ]
    for c in candidates:
        if c.exists():
            return c
    # fallback: glob
    hits = sorted(gt_dir.glob("*GTconvres_135nm.tif"))
    if hits:
        return hits[0]
    return candidates[0]  # return expected path for debugging


def _collect_pattern_items(sim_dir: Path) -> List[PatternItem]:
    sim_id = sim_dir.name.split("_", 1)[0]
    items: List[PatternItem] = []

    for pat_dir in sorted(sim_dir.iterdir()):
        if not _is_pattern_dir(pat_dir):
            continue

        pattern_id = pat_dir.name.split("_", 1)[0]

        gt_path = _find_gt_file(pat_dir)
        noisy_dir = pat_dir / "Noisy"

        sofi_root = pat_dir / "_SOFI_Results_fwhm2.7"
        sofi_run_dir = _pick_sofi_run_dir(sofi_root) or Path("")

        sofi_c2_files: List[Path] = []
        if sofi_run_dir and sofi_run_dir.exists():
            for f in sorted(sofi_run_dir.iterdir()):
                if not f.is_file():
                    continue
                name = f.name.lower()
                if name.endswith("_noisyvid__mctsofi_c2.tif") and not name.startswith("04_"):
                    sofi_c2_files.append(f)

        items.append(
            PatternItem(
                sim_id=sim_id,
                pattern_id=pattern_id,
                pattern_dir=pat_dir,
                gt_path=gt_path,
                noisy_dir=noisy_dir,
                sofi_run_dir=sofi_run_dir,
                sofi_c2_files=sofi_c2_files,
            )
        )

    return items


def _write_csv(path: Path, items: List[PatternItem]) -> None:
    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow([
            "sim_id",
            "pattern_id",
            "pattern_dir",
            "gt_path",
            "noisy_dir",
            "sofi_run_dir",
            "sofi_c2_files",
        ])
        for it in items:
            w.writerow([
                it.sim_id,
                it.pattern_id,
                str(it.pattern_dir),
                str(it.gt_path),
                str(it.noisy_dir),
                str(it.sofi_run_dir),
                ";".join(str(x) for x in it.sofi_c2_files),
            ])

def create_fixed_splits() -> None:
    if not DATA_DIR.exists():
        raise FileNotFoundError(f"DATA_DIR does not exist: {DATA_DIR}")

    SPLITS_DIR.mkdir(parents=True, exist_ok=True)

    train_csv = SPLITS_DIR / "train.csv"
    val1_csv = SPLITS_DIR / "val1.csv"
    val2_csv = SPLITS_DIR / "val2.csv"
    test_csv = SPLITS_DIR / "test.csv"

    if not OVERWRITE_SPLITS and (
        train_csv.exists() or val1_csv.exists() or val2_csv.exists() or test_csv.exists()
    ):
        print(f"Splits already exist in: {SPLITS_DIR}")
        print("Set OVERWRITE_SPLITS = True to regenerate.")
        return

    sim07_dir = _find_sim_dir(DATA_DIR, "07")
    sim08_dir = _find_sim_dir(DATA_DIR, "08")

    train_items = _collect_pattern_items(sim07_dir)
    sim08_items = _collect_pattern_items(sim08_dir)

    if len(sim08_items) < 2:
        raise RuntimeError("Need at least 2 patterns in sim 08 to split val/test.")

    rng = random.Random(SEED)
    rng.shuffle(sim08_items)

    half = len(sim08_items) // 2
    val_items = sim08_items[:half]
    test_items = sim08_items[half:]

    _write_csv(train_csv, train_items)
    _write_csv(val1_csv, val_items)
    _write_csv(val2_csv, val_items)   # <-- EXACT SAME as val1
    _write_csv(test_csv, test_items)

    print("Splits written to:", SPLITS_DIR)
    print(f"Train patterns (07): {len(train_items)}")
    print(f"Val1 patterns  (08): {len(val_items)}")
    print(f"Val2 patterns  (08): {len(val_items)}")
    print(f"Test patterns  (08): {len(test_items)}")



# Run split creation
create_fixed_splits()


Splits written to: C:\Users\ntpar\Downloads\SN2N_Capstone\splits
Train patterns (07): 100
Val1 patterns  (08): 30
Val2 patterns  (08): 30
Test patterns  (08): 30


## 2) Create preprocessed folders (train/val/test)

Notes:
- This expects `SN2N_resampling_fourier_RL_deconvolution_augmentation_py.py` (or module) to be importable.
- Train/val: full pipeline + SN2N sample creation
- Test: RL deconvolution + percentile normalization only (no checkerboard splitting)


In [ ]:
# Import pipeline helpers (kept minimal: only what is used below)
try:
    from SN2N_resampling_fourier_RL_deconvolution_augmentation_py import (
        run_pipeline,
        make_sn2n_sample,
        save_sn2n_sample,
        rl_deconvolve,
        FWHM,
        RL_ITERS,
        FLOOR,
    )
except ImportError as e:
    raise ImportError(
        "Could not import SN2N_resampling_fourier_RL_deconvolution_augmentation_py.\n"
        "Make sure the file is in the same folder as this notebook, or installed as a module."
    ) from e


def _parse_semicolon_list(s: str) -> List[str]:
    s = (s or "").strip()
    if not s:
        return []
    return [x.strip() for x in s.split(";") if x.strip()]


def _read_split_csv(csv_path: Path) -> List[Dict[str, str]]:
    with csv_path.open("r", newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def _make_psf_from_fwhm(fwhm: float) -> np.ndarray:
    sigma = fwhm / 2.355
    psf_size = int(np.ceil(sigma * 6))
    if psf_size % 2 == 0:
        psf_size += 1
    ax = np.arange(psf_size) - psf_size // 2
    xx, yy = np.meshgrid(ax, ax)
    psf = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    psf /= psf.sum()
    return psf


def _percentile_normalize(img: np.ndarray, p_low: float, p_high: float, eps: float = 1e-6) -> np.ndarray:
    img = img.astype(np.float32)
    lo = np.percentile(img, p_low)
    hi = np.percentile(img, p_high)
    if (hi - lo) < eps:
        hi = lo + eps
    out = (img - lo) / (hi - lo)
    return np.clip(out, 0.0, 1.0).astype(np.float32)


def process_split(split_name: str) -> None:
    split_name = split_name.lower()
    split_csv = SPLITS_DIR / f"{split_name}.csv"
    if not split_csv.exists():
        raise FileNotFoundError(f"Missing split file: {split_csv}")

    rows = _read_split_csv(split_csv)
    if VERBOSE:
        print(f"\n[{split_name}] Loaded {len(rows)} pattern-rows from {split_csv}")

    out_split_dir = OUT_DIR / split_name
    out_split_dir.mkdir(parents=True, exist_ok=True)

    written = skipped = missing = 0

    # Treat val2 as an additional test-like split
    is_test_like = split_name in ("test", "val2")

    # Precompute PSF once for test-like splits
    psf = _make_psf_from_fwhm(FWHM) if is_test_like else None

    for r in rows:
        sim_id = r.get("sim_id", "NA")
        pattern_id = r.get("pattern_id", "NA")
        c2_list = _parse_semicolon_list(r.get("sofi_c2_files", ""))
        if not c2_list:
            continue

        for c2_path_str in c2_list:
            c2_path = Path(c2_path_str)
            if not c2_path.exists():
                missing += 1
                if VERBOSE:
                    print(f"  [MISSING] {c2_path}")
                continue

            stem = c2_path.stem
            if not stem.startswith(f"{sim_id}_{pattern_id}_"):
                stem = f"{sim_id}_{pattern_id}_{stem}"

            suffix = "_RL.tif" if is_test_like else "_SN2N.tif"
            out_path = out_split_dir / f"{stem}{suffix}"

            if out_path.exists() and not OVERWRITE_PREPROCESSED:
                skipped += 1
                continue

            if is_test_like:
                img = tiff.imread(c2_path)
                if img.ndim > 2:
                    img = img[0]

                img_rl = rl_deconvolve(img, psf, iters=RL_ITERS, floor=FLOOR)
                img_norm = _percentile_normalize(img_rl, P_LOW, P_HIGH)
                save_sn2n_sample(img_norm, out_path)
                written += 1
                continue

            # Train/Val1: physics-aware pipeline (same as your current "val")
            if np.random.rand() < P_BASIC_AUG:
                basic_aug_mode = np.random.randint(0, 8)
            else:
                basic_aug_mode = None

            n_repeats = 1 + max(0, P2P_UP)
            for rep in range(n_repeats):
                res = run_pipeline(
                    str(c2_path),
                    p2p_mode=P2P_MODE,
                    p2p_patch_size=P2P_PATCH_SIZE,
                    use_sliding=USE_SLIDING,
                    slide_patch_size=SLIDE_PATCH_SIZE,
                    slide_stride=SLIDE_STRIDE,
                    basic_aug_mode=basic_aug_mode,
                )

                sample = make_sn2n_sample(
                    res["left_rl"],
                    res["right_rl"],
                    p_low=P_LOW,
                    p_high=P_HIGH,
                )

                if P2P_UP > 0:
                    out_path_rep = out_path.with_name(f"{out_path.stem}_p2p{rep}.tif")
                else:
                    out_path_rep = out_path

                if out_path_rep.exists() and not OVERWRITE_PREPROCESSED:
                    skipped += 1
                    continue

                save_sn2n_sample(sample, out_path_rep)
                written += 1

                if VERBOSE and written % 25 == 0:
                    print(f"  wrote {written} samples so far...")

    print(f"[{split_name}] Done. wrote={written}, skipped={skipped}, missing_inputs={missing}")
    print(f"Outputs in: {out_split_dir}\n")


def run_all_splits() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # Run the new split set; (optional) keep backward compat if val.csv exists
    for split in ("train", "val1", "val2", "test"):
        if (SPLITS_DIR / f"{split}.csv").exists():
            process_split(split)

# Run preprocessing
run_all_splits()



[train] Loaded 100 pattern-rows from C:\Users\ntpar\Downloads\SN2N_Capstone\splits\train.csv
  wrote 25 samples so far...
  wrote 50 samples so far...
  wrote 75 samples so far...
  wrote 100 samples so far...
  wrote 125 samples so far...
  wrote 150 samples so far...
  wrote 175 samples so far...
  wrote 200 samples so far...
  wrote 225 samples so far...
  wrote 250 samples so far...
  wrote 275 samples so far...
  wrote 300 samples so far...
  wrote 325 samples so far...
  wrote 350 samples so far...
  wrote 375 samples so far...
  wrote 400 samples so far...
  wrote 425 samples so far...
  wrote 450 samples so far...
  wrote 475 samples so far...
  wrote 500 samples so far...
  wrote 525 samples so far...
  wrote 550 samples so far...
  wrote 575 samples so far...
  wrote 600 samples so far...
  wrote 625 samples so far...
  wrote 650 samples so far...
  wrote 675 samples so far...
  wrote 700 samples so far...
  wrote 725 samples so far...
  wrote 750 samples so far...
  wrote 7

## 3) Map test outputs to GT files

Uses the test filenames to reconstruct the corresponding GT path.


In [ ]:
def get_gt_path_from_test_filename(test_filename: str, data_dir: Path = DATA_DIR) -> Optional[Path]:
    """Map an output filename (test/val) to its corresponding ground truth path."""
    name = Path(test_filename).stem

    parts = name.split("_", 2)
    if len(parts) < 2:
        print(f"Could not parse filename: {test_filename}")
        return None

    sim_id, pattern_id = parts[0], parts[1]

    sim_matches = [d for d in data_dir.iterdir() if d.is_dir() and d.name.startswith(f"{sim_id}_")]
    if not sim_matches:
        print(f"No simulation folder found for sim_id={sim_id} in {data_dir}")
        return None
    sim_dir = sorted(sim_matches, key=lambda x: x.stat().st_mtime, reverse=True)[0]

    pattern_matches = [
        d for d in sim_dir.iterdir()
        if d.is_dir() and d.name.startswith(f"{pattern_id}_") and d.name.endswith("_Training_")
    ]
    if not pattern_matches:
        print(f"No pattern folder found for pattern_id={pattern_id} in {sim_dir}")
        return None
    pattern_dir = pattern_matches[0]

    return _find_gt_file(pattern_dir)  # assumes you have this from your split script


def create_split_gt_mapping(split_dir: Path, data_dir: Path = DATA_DIR) -> pd.DataFrame:
    tif_files = sorted(split_dir.glob("*.tif"))
    records = []

    for p in tif_files:
        gt_path = get_gt_path_from_test_filename(p.name, data_dir)
        records.append({
            "split": split_dir.name,
            "file": p.name,
            "path": str(p),
            "gt_path": str(gt_path) if gt_path else None,
            "gt_exists": bool(gt_path and gt_path.exists()),
        })

    df = pd.DataFrame(records)
    print(f"[{split_dir.name}] Found {len(df)} files")
    if len(df) > 0:
        print(f"  - With valid GT: {df['gt_exists'].sum()}")
        print(f"  - Missing GT: {(~df['gt_exists']).sum()}")
    return df


# ---- Run for the splits you want ----
mappings = []

for split_name in ("test", "val2"): 
    split_dir = OUT_DIR / split_name
    if split_dir.exists():
        df = create_split_gt_mapping(split_dir, DATA_DIR)
        mappings.append(df)

        out_csv = OUT_DIR / f"{split_name}_gt_mapping.csv"
        df.to_csv(out_csv, index=False)
        print(f"Mapping saved to: {out_csv}\n")
    else:
        print(f"Directory not found, skipping: {split_dir}")


[test] Found 300 files
  - With valid GT: 300
  - Missing GT: 0
Mapping saved to: C:\Users\ntpar\Downloads\SN2N_Capstone\splits\preprocessed-p2p\test_gt_mapping.csv

[val2] Found 300 files
  - With valid GT: 300
  - Missing GT: 0
Mapping saved to: C:\Users\ntpar\Downloads\SN2N_Capstone\splits\preprocessed-p2p\val2_gt_mapping.csv

